# Credit Card Fraud Detection using Support Vector Machines

**A code walkthrough.**

This notebook trains an SVM classifier on the Kaggle Credit Card Fraud dataset and evaluates it.

## 1. SVM in brief

A **Support Vector Machine** is a supervised classifier that finds the decision boundary (a hyperplane) that maximally separates two classes. The points closest to the boundary are called **support vectors** — they are the only points that define the boundary.

- With a **linear kernel**, SVM draws a straight boundary.
- With an **RBF (radial basis function) kernel**, SVM maps data into a higher-dimensional space so it can carve out **non-linear** boundaries. This is what we use here.

**Why SVM fits fraud detection:**
- Fraud data has complex, non-linear patterns → RBF kernel handles that.
- Fraud is rare (~0.17% of transactions) → SVM's `class_weight='balanced'` option lets us tell the model to care more about the minority class.
- Features are numeric and already PCA-transformed (`V1`…`V28`) → SVM works well on scaled numeric features.

**Where SVM struggles:**
- Slow on very large datasets (training scales ~O(n²)) → we subsample the legit class.
- Sensitive to feature scale → we must standardize before training.

## 2. Imports and configuration

We pull in `SVC` (the SVM classifier), `StandardScaler` (for feature scaling), and the metrics we'll use for evaluation. The config block sets our hyperparameters in one place so they're easy to tweak.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_score, recall_score, f1_score,
)

DATA_PATH = os.path.join("archive", "creditcard.csv")
N_LEGIT_SAMPLES = 6000   # subsample legit class to keep SVM training fast
KERNEL = "rbf"           # non-linear kernel
C_VALUE = 1.0            # regularization strength (smaller = wider margin)
GAMMA = "scale"          # RBF kernel width (auto-scaled by sklearn)
RANDOM_STATE = 42

## 3. Load and prepare the data

The raw dataset has ~284,807 transactions but only ~492 frauds — that's a **0.17% fraud rate**. Training SVM on all 284k rows would take forever, so we:

1. Keep **all** fraud cases (they're rare and precious).
2. Randomly sample 6,000 legit transactions.
3. Shuffle the combined set.

This gives us ~6,500 rows — still imbalanced (~7.5% fraud), which is realistic, but tractable for SVM.

In [ ]:
df = pd.read_csv(DATA_PATH)

fraud = df[df["Class"] == 1]
legit = df[df["Class"] == 0].sample(n=N_LEGIT_SAMPLES, random_state=RANDOM_STATE)
data = pd.concat([fraud, legit]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X = data.drop(columns=["Class"]).values   # features
y = data["Class"].values                  # labels: 0 = legit, 1 = fraud

print(f"Full dataset: {len(df):,} rows")
print(f"  Fraud cases: {(df['Class'] == 1).sum()}")
print(f"  Legit cases: {(df['Class'] == 0).sum()}")
print(f"  Fraud rate:  {(df['Class'] == 1).mean() * 100:.3f}%\n")
print(f"Training subset: {len(data)} rows ({(y == 1).sum()} fraud + {(y == 0).sum()} legit)")

## 4. Train/test split + feature scaling

Two things happen here:

1. **`train_test_split` with `stratify=y`** — ensures both the train and test sets have the same fraud-to-legit ratio. Without `stratify`, a random split could accidentally put most fraud cases in one side.
2. **`StandardScaler`** — SVM with RBF kernel computes distances between points. If one feature has values in thousands and another in 0–1, the large one dominates. Scaling fixes that by giving every feature mean 0 and std 1.

**Important:** we `fit` the scaler on the **training data only**, then `transform` both train and test. Fitting on test data would leak information.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train size: {len(X_train)}  |  Test size: {len(X_test)}")
print(f"Train fraud ratio: {y_train.mean()*100:.2f}%  |  Test fraud ratio: {y_test.mean()*100:.2f}%")

## 5. Train the SVM — the core cell

This is the heart of the project. Three hyperparameters decide everything:

- **`kernel='rbf'`** — use the Radial Basis Function kernel so the model can learn curved, non-linear boundaries between fraud and legit.
- **`C=1.0`** — regularization. High C = model tries harder to classify every training point correctly (risk of overfitting). Low C = wider margin, more tolerance for mistakes. `1.0` is a balanced default.
- **`gamma='scale'`** — controls how far the influence of a single training example reaches. `'scale'` lets sklearn pick a sensible value based on feature variance.
- **`class_weight='balanced'`** — **this is the key line for fraud detection**. It tells SVM to penalize misclassifying the rare fraud class proportionally more. Without it, the model would happily predict "legit" for everything and still get 92%+ accuracy.

After `.fit()`, `model.support_vectors_` holds the actual points that define the decision boundary.

In [ ]:
model = SVC(
    kernel=KERNEL,
    C=C_VALUE,
    gamma=GAMMA,
    class_weight="balanced",
)

print(f"Training SVM (kernel={KERNEL}, C={C_VALUE}, gamma={GAMMA})...")
model.fit(X_train_s, y_train)
print(f"Support vectors used: {model.support_vectors_.shape[0]} / {len(X_train)} training points")

## 6. Evaluate the model

For an imbalanced problem like fraud, **accuracy is misleading** — a dumb "predict legit always" classifier would score ~92% accuracy. So we use:

- **Precision** — of all transactions we flagged as fraud, how many actually were? (High precision = few false alarms.)
- **Recall** — of all actual fraud cases, how many did we catch? (High recall = few frauds slip through.)
- **F1** — harmonic mean of precision and recall, balances both.
- **ROC-AUC** — how well the model ranks fraud above legit across all thresholds. 1.0 = perfect, 0.5 = random.
- **Confusion matrix** — raw counts of TP / TN / FP / FN.

`model.decision_function` gives the signed distance from the boundary (used for the ROC curve), while `model.predict` gives the final 0/1 label.

In [ ]:
y_pred  = model.predict(X_test_s)
y_score = model.decision_function(X_test_s)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)
fpr, tpr, _ = roc_curve(y_test, y_score)
roc_auc   = auc(fpr, tpr)

print("=" * 48)
print("Test Results")
print("=" * 48)
print(f"Precision (fraud): {precision:.3f}")
print(f"Recall    (fraud): {recall:.3f}")
print(f"F1 score:          {f1:.3f}")
print(f"ROC AUC:           {roc_auc:.3f}\n")
print("Confusion matrix:")
print(f"                 Pred Legit   Pred Fraud")
print(f"  True Legit     {tn:10d}   {fp:10d}")
print(f"  True Fraud     {fn:10d}   {tp:10d}\n")
print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))

## 7. Visualize — confusion matrix and ROC curve

The heatmap makes the confusion matrix easier to read at a glance. The ROC curve shows the trade-off between catching fraud (TPR) and false alarms (FPR) across every possible threshold — the area under it (AUC) summarizes overall ranking quality.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Pred Legit", "Pred Fraud"],
            yticklabels=["True Legit", "True Fraud"], ax=axes[0])
axes[0].set_title("Confusion Matrix")

axes[1].plot(fpr, tpr, color="#2F7FBF", lw=2, label=f"AUC = {roc_auc:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="#aaa")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

## 8. (Optional) Tuning C with cross-validation

Instead of picking `C=1.0` by guess, we can try a few values and use **5-fold stratified cross-validation** to see which gives the best F1. Each fold preserves the fraud ratio (that's what *stratified* means).

In [ ]:
scaler_full = StandardScaler().fit(X)
X_s = scaler_full.transform(X)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("5-fold CV for C (scoring = F1)")
print("-" * 40)
for C in (0.1, 1.0, 10.0):
    m = SVC(kernel=KERNEL, C=C, gamma=GAMMA, class_weight="balanced")
    scores = cross_val_score(m, X_s, y, cv=cv, scoring="f1", n_jobs=-1)
    print(f"  C = {C:>6}: F1 = {scores.mean():.3f}  (std {scores.std():.3f})")

## 9. Takeaways

- RBF-kernel SVM with `class_weight='balanced'` handles imbalanced fraud data well.
- Feature scaling is **mandatory** for SVM — skip it and performance collapses.
- Precision/recall/F1/ROC-AUC are the right metrics here, not accuracy.
- The trained model uses only ~a few hundred **support vectors** out of thousands of training points — that's the data-efficient core of SVM.

An interactive Streamlit version of this pipeline lives in `app.py` (`streamlit run app.py`).